In [1]:
import requests
import json
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path

In [5]:
load_dotenv()

username_env = os.getenv("username")
password_env = os.getenv("password")

In [6]:

def get_access_token(username, password, token_url):
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
    }
    data = {
          'username': username_env,
          'password': password_env,
          'grant_type': "password",
          'client_id': "acled"
    }

    response = requests.post(token_url, headers=headers, data=data)

    if response.status_code == 200:
        token_data = response.json()
        return token_data['access_token']
    else:
        raise Exception(f"Failed to get access token: {response.status_code} {response.text}")


In [7]:
my_token = get_access_token(
    username=username_env,
    password=password_env,
    token_url="https://acleddata.com/oauth/token",
)

In [8]:
dir_raw = Path('../data/raw')
dir_raw.mkdir(exist_ok=True)
dir_processed = Path('../data/processed')
dir_processed.mkdir(exist_ok=True)

The below code set ups an API call. The structure of the response = requests.get (including the headers) is from ACLED's documentation. 

The code is meant to run until it reaches the end of the dataset. A counter has been created (iteration) to ensure that the pagination works - this is added to the end of the url and updates for each cycle. There are some print statements in the code that should help keep track of progress and debug issues if you encounter any. The break statement takes the content from the key 'data' (which has been checked manually through iterations beforehand) and stops when the number of returned records are less than 5000, as per ACLED's API documentation.

Each page is loaded directly to the raw folder and saved as a json file. These will be loaded and concatenated into a single CSV file in NB02.

In [9]:
def API_connection(url):

    iteration = 0

    while True: 

        iteration += 1

        base_url = url + str(iteration)

        response = requests.get(
            base_url,
            headers={"Authorization": f"Bearer {my_token}", "Content-Type": "application/json"},)

        status = response.status_code

        if status == 200:
            print(f"Request {iteration} successful")
            
            data = response.json()

            filename = (f"Syria_Conflict_Events_{iteration}.json")

            with open(f"{dir_raw}/{filename}", "w") as f:
                json.dump(data, f, indent=4) 
            
            print(f"dataset {iteration} saved as json file")

            print(f"key 'data' inside dataset {iteration} is {type(data['data'])} and has {len(data['data'])} entries")

        else:
            print(f"connection failed, status code {status}")

        if len(data['data']) < 5000: 
            break

    return print(f"\ndata collection completed")

    

Specify the fields, that will be used in the url, and run the function.

In [10]:
fields = "country|event_id_cnty|event_date|time_precision|event_type|disorder_type|sub_event_type|actor1|assoc_actor_1|inter1|actor2|assoc_actor_2|inter2|civilian_targeting|admin1|admin3|location|geo_precision|source|fatalities|tags|population_5km"

In [11]:
url = (f"https://acleddata.com/api/acled/read?_format=json&country=Syria&event_date=2024-12-08|2026-07-26&event_date_where=BETWEEN&fields={fields}&page=")

In [12]:
API_connection(url)

Request 1 successful
dataset 1 saved as json file
key 'data' inside dataset 1 is <class 'list'> and has 5000 entries
Request 2 successful
dataset 2 saved as json file
key 'data' inside dataset 2 is <class 'list'> and has 5000 entries
Request 3 successful
dataset 3 saved as json file
key 'data' inside dataset 3 is <class 'list'> and has 5000 entries
Request 4 successful
dataset 4 saved as json file
key 'data' inside dataset 4 is <class 'list'> and has 1845 entries

data collection completed
